# FingerBall Stage 2J-1B | Finger-specific QC & interactive 3D comparison

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_Stage2J1B_Detailed_QC_3D_Viewer.ipynb)

**目的：** Stage 2J-1で成功した推論結果を再利用し、リリース前後の**示指・中指（MCP/PIP/DIP/TIP）**の2D誤差を詳しく検証する。MediaPipe / WiLoR-miniの**モデル別推定3D**は、360°回転＋フレーム操作可能なHTMLビューアで比較する。

- **推論は再実行しません**（WiLoRの環境構築も不要）。
- Stage 2Hで目視確認済みのXYは一切変更しません。
- 参照データを使ったROI補助条件は独立した検出性能ではありません。
- MediaPipeとWiLoRの推定3Dは、座標系とスケールが異なります。ビューアでは手首原点・手掌幅で**表示目的に正規化**します。**真の3D精度比較ではありません**。
- 無検出や遮蔽は欠損のまま残します。勝手な時系列補間はしません。

**Stage 2J-2 SOMA統合前のQCゲート**を作るNotebookです。

## 0. Google Driveとライブラリ

In [ ]:
%pip -q install "pandas>=2.0" "numpy>=1.24" "plotly>=5.18" "matplotlib>=3.7" "opencv-python-headless>=4.8"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from IPython.display import display,HTML
import pandas as pd,numpy as np,matplotlib.pyplot as plt
import cv2,json,hashlib,html
import plotly.graph_objects as go
import plotly.io as pio

## 1. 入力・検証・データ保全

Stage 2J-1で出力された共通CSVを指定します。モデル別の再推論は行いません。

In [ ]:
PROJECT=Path('/content/drive/MyDrive/母指回内トラッキング')
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
INPUT_CSV=PROJECT/'FingerBall_v2_stage2j1_results'/VIDEO_STEM/'stage2j1_common_21_landmarks_with_provenance.csv'
SOURCE_VIDEO=PROJECT/(VIDEO_STEM+'.mp4')  # optional
OUT=PROJECT/'FingerBall_v2_stage2j1b_results'/VIDEO_STEM
FRAMES=list(range(185,199))
DETAIL_FRAMES=list(range(188,194))
RELEASE_REFERENCE=190
METHODS=['mediapipe_full','mediapipe_manual_roi','wilor_full_detector','wilor_manual_bbox']
FINGER_IDS={'Index':(5,6,7,8),'Middle':(9,10,11,12)}
FINGER_JOINTS=('MCP','PIP','DIP','TIP')
if not INPUT_CSV.is_file():raise FileNotFoundError('Stage 2J-1 CSVが見つかりません: '+str(INPUT_CSV))
OUT.mkdir(parents=True,exist_ok=True)
source_digest=hashlib.sha256(INPUT_CSV.read_bytes()).hexdigest()
data=pd.read_csv(INPUT_CSV)
required={'frame','method','landmark_id','stage2h_x_px','stage2h_y_px',
          'x_px','y_px','model3d_x','model3d_y','model3d_z',
          'status_stage2h','detected','assisted_by_stage2h_roi'}
assert required.issubset(data.columns),required-set(data.columns)
if data.duplicated(['frame','method','landmark_id']).any():raise ValueError('CSV duplicate frame/method/landmark')
subset=data[data.frame.isin(FRAMES)].copy()
for f in FRAMES:
    for m in METHODS+['stage2h_reference']:
        if subset[(subset.frame==f)&(subset.method==m)].landmark_id.nunique()!=21:
            raise ValueError(f'21 points missing: frame={f} method={m}')
data=subset
print('Input SHA256:',source_digest,'rows:',len(data))
print('Detected frames:',data[data.method.isin(METHODS)].groupby('method').detected.sum()/21)

## 2. 指・関節ごとの誤差集計（手動点のみも別集計）

各ランドマークの**XY Euclidean error**を再計算。欠損と遮蔽は除外。全フレームで検出できていないモデルは欠測の多さも必ず表示します。

In [ ]:
eligible=data[data.method.isin(METHODS)].copy()
eligible['ref_valid']=eligible[['stage2h_x_px','stage2h_y_px']].notna().all(axis=1)
eligible['pred_valid']=eligible[['x_px','y_px']].notna().all(axis=1)&eligible.detected.fillna(False)
eligible['evaluated']=eligible.ref_valid&eligible.pred_valid&(~eligible.status_stage2h.isin(['occluded','missing']))
eligible['error_px']=np.where(eligible.evaluated,np.hypot(
    eligible.x_px-eligible.stage2h_x_px,
    eligible.y_px-eligible.stage2h_y_px),np.nan)
joint_lookup={idx:(finger,joint) for finger,ids in FINGER_IDS.items()
              for idx,joint in zip(ids,FINGER_JOINTS)}
eligible['finger']=eligible.landmark_id.map(lambda x:joint_lookup.get(x,('Other','Other'))[0])
eligible['joint']=eligible.landmark_id.map(lambda x:joint_lookup.get(x,('Other','Other'))[1])
finger=eligible[eligible.finger.isin(FINGER_IDS)].copy()
finger.to_csv(OUT/'stage2j1b_jointwise_long.csv',index=False)
def rmse(v):
    v=np.asarray(pd.Series(v).dropna(),float)
    return np.sqrt(np.mean(v*v)) if len(v) else np.nan
agg=(finger.groupby(['method','finger','joint'],as_index=False)
    .agg(n_evaluable=('error_px','count'),
         n_observation_rows=('error_px','size'),
         rmse_px=('error_px',rmse),
         median_px=('error_px','median'),
         max_error_px=('error_px','max')))
manual=(finger[finger.status_stage2h.eq('manual_visible')]
    .groupby(['method','finger','joint'],as_index=False)
    .agg(n_manual=('error_px','count'),manual_rmse_px=('error_px',rmse)))
agg=agg.merge(manual,on=['method','finger','joint'],how='left')
agg.to_csv(OUT/'stage2j1b_jointwise_summary.csv',index=False)
frameagg=(finger.groupby(['method','frame','finger'],as_index=False)
    .agg(n_evaluable=('error_px','count'),rmse_px=('error_px',rmse)))
frameagg.to_csv(OUT/'stage2j1b_per_frame_finger_rmse.csv',index=False)
display(agg.round(2))
print('Reminder: small RMSE on only a few detected landmarks does not imply good detection rate')

## 3. 示指・中指の時間変化と関節別誤差ヒートマップ

In [ ]:
for fname in FINGER_IDS:
    t=frameagg[frameagg.finger==fname]
    fig,ax=plt.subplots(figsize=(11,4.5))
    for method in METHODS:
        z=t[t.method==method]
        ax.plot(z.frame,z.rmse_px,marker='o',label=method)
    ax.axvline(RELEASE_REFERENCE,color='black',linestyle='--',alpha=.6)
    ax.set(title=f'{fname} hand landmarks 2D error to Stage 2H (detected only)',
           xlabel='Original video frame',ylabel='RMSE pixels')
    ax.legend(fontsize=8);ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(OUT/f'stage2j1b_{fname.lower()}_rmse_timeseries.png',dpi=160);plt.show()
for method in METHODS:
    h=agg[agg.method==method].copy()
    mat=h.pivot(index='finger',columns='joint',values='rmse_px').reindex(
        index=['Index','Middle'],columns=list(FINGER_JOINTS))
    fig,ax=plt.subplots(figsize=(6.5,2.8))
    im=ax.imshow(mat.values,aspect='auto',cmap='viridis',vmin=0)
    ax.set_xticks(np.arange(4),mat.columns)
    ax.set_yticks(np.arange(2),mat.index)
    ax.set_title(method+' | 2D RMSE (pixels)')
    for i in range(2):
        for j in range(4):
            v=mat.values[i,j]
            ax.text(j,i,f'{v:.1f}' if np.isfinite(v) else 'NA',ha='center',
                    va='center',color='white',fontsize=10)
    fig.colorbar(im,ax=ax,label='RMSE px')
    fig.tight_layout()
    fig.savefig(OUT/f'stage2j1b_joint_heatmap_{method}.png',dpi=160);plt.show()

## 4. 188〜193F 手部の拡大2D比較

1フレームずつ、手動確定座標（黄色）とモデル推定座標（シアン）を描き、**指先のズレを線で表示**します。元動画が見つからない場合も2D座標の重畳を表示します。

In [ ]:
EDGE_GROUPS={'Thumb':[0,1,2,3,4],'Index':[0,5,6,7,8],
             'Middle':[0,9,10,11,12],'Ring':[0,13,14,15,16],
             'Little':[0,17,18,19,20]}
def coords(f,m,columns):
    a=data[(data.frame==f)&(data.method==m)].set_index('landmark_id').reindex(range(21))
    return a[columns].to_numpy(float)
video=cv2.VideoCapture(str(SOURCE_VIDEO)) if SOURCE_VIDEO.is_file() else None
if video and not video.isOpened():video.release();video=None
def graph_skel(ax,points,color,lw=1.5):
    for group,ids in EDGE_GROUPS.items():
        for a,b in zip(ids[:-1],ids[1:]):
            if np.isfinite(points[[a,b]]).all():
                ax.plot(points[[a,b],0],points[[a,b],1],color=color,lw=lw)
    mask=np.isfinite(points).all(axis=1)
    ax.scatter(points[mask,0],points[mask,1],color=color,s=8)
for f in DETAIL_FRAMES:
    obs=coords(f,'stage2h_reference',['stage2h_x_px','stage2h_y_px'])
    fig,axes=plt.subplots(1,4,figsize=(18,4.8),sharex=True,sharey=True)
    frame_image=None
    if video:
        video.set(cv2.CAP_PROP_POS_FRAMES,f);ok,bgr=video.read()
        if ok:frame_image=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    mid=np.nanmedian(obs,axis=0)
    for ax,m in zip(axes,METHODS):
        if frame_image is not None:ax.imshow(frame_image)
        else:ax.set_ylim(648,0)
        pred=coords(f,m,['x_px','y_px'])
        graph_skel(ax,obs,'#ffb545',2)
        graph_skel(ax,pred,'#00f5e9',1.5)
        selected=list(FINGER_IDS['Index']+FINGER_IDS['Middle'])
        for j in selected:
            if np.isfinite(obs[j]).all() and np.isfinite(pred[j]).all():
                ax.plot([obs[j,0],pred[j,0]],[obs[j,1],pred[j,1]],
                        color='#f97316',lw=1,alpha=.8)
        ax.set(xlim=(mid[0]-105,mid[0]+105),
               ylim=(mid[1]+105,mid[1]-105))
        ax.set_title(m,fontsize=9);ax.set_xticks([]);ax.set_yticks([])
    fig.suptitle(f'Frame {f}: Stage2H orange / model cyan / index-middle error links')
    fig.tight_layout()
    fig.savefig(OUT/f'stage2j1b_frame_{f}_closeup.png',dpi=150)
    plt.show()
if video:video.release()

## 5. 360°回転・時間軸付き3D骨格ビューア（HTML単独共有可）

MediaPipe / WiLoR-miniの**各モデル独立の形状正規化**で比較します。ボタンとスライダーはPlotlyキャンバス外の固定HTMLバーとして配置し、表示領域に埋もれないようにしています。遮蔽・未検出は空表示。ボタンでモデル切替、フレーム操作、Play/Pause、ドラッグ回転できます。

⚠️ 見かけの3D骨格が滑らかでも、それは単眼3Dの正しさを証明しません。

In [ ]:
# Prepare small model-specific JSON for a standalone Plotly HTML.
MODEL_LABELS={'mediapipe_full':'MediaPipe full',
 'mediapipe_manual_roi':'MediaPipe manual ROI',
 'wilor_full_detector':'WiLoR full',
 'wilor_manual_bbox':'WiLoR manual bbox'}
bones=[(a,b) for ids in EDGE_GROUPS.values() for a,b in zip(ids[:-1],ids[1:])]
viewer_payload={}
for m in METHODS:
    viewer_payload[m]={}
    for f in FRAMES:
        xyz=coords(f,m,['model3d_x','model3d_y','model3d_z'])
        if not np.isfinite(xyz).all():viewer_payload[m][str(f)]=None;continue
        xyz=xyz-xyz[0]
        scale=np.linalg.norm(xyz[9])
        if not np.isfinite(scale) or scale<1e-8:
            viewer_payload[m][str(f)]=None;continue
        xyz=xyz/scale
        viewer_payload[m][str(f)]=xyz.round(5).tolist()
viewer_html=OUT/'FingerBall_Stage2J1B_3D_Comparison_360.html'
# pio.to_html supplies a complete local plotly.js library, so no network is needed.
initial=go.Figure(data=[go.Scatter3d(x=[],y=[],z=[],mode='lines',
      line={'color':'#67e8f9','width':7},hoverinfo='skip'),
      go.Scatter3d(x=[],y=[],z=[],mode='markers',
      marker={'size':4,'color':'#fbbf24'},hoverinfo='skip')])
initial.update_layout(template='plotly_dark',
    title='FingerBall — normalized 3D joint estimates (not ground truth)',
    height=650,margin=dict(l=0,r=0,t=75,b=10),showlegend=False,
    scene=dict(aspectmode='cube',uirevision='camera-fixed',
         xaxis={'range':[-3,3]},yaxis={'range':[-3,3]},zaxis={'range':[-3,3]}))
figure_html=pio.to_html(initial,full_html=False,include_plotlyjs=True,
      div_id='fb3d',config={'responsive':True,'scrollZoom':True,'displaylogo':False})
template=r"""<!doctype html><html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>FingerBall: 3D model comparison</title>
<style>
body{background:#0b1220;color:#e2e8f0;font:15px system-ui,sans-serif;margin:0}
main{padding:14px;max-width:1300px;margin:auto}
.toolbar{display:flex;flex-wrap:wrap;gap:12px;align-items:center;padding:14px;
background:#1e293b;border-radius:10px;margin:12px 0}
button,select{background:#334155;color:#fff;border:1px solid #94a3b8;
border-radius:7px;padding:9px;font-size:15px}
input[type=range]{flex:1;min-width:200px}
#fb3d{width:100%;min-height:600px}
small{color:#cbd5e1}
</style></head><body><main>
<h2>FingerBall — Model 3D Skeleton Viewer</h2>
<small>3D coordinates independently normalized to wrist origin and wrist–middle MCP length. NOT calibrated physical 3D.</small>
<div class="toolbar">
<label for="method">Model</label><select id="method">__OPTIONS__</select>
<button id="play">▶ Play</button><button id="pause">Ⅱ Pause</button>
<label for="frame">Frame <strong id="ftext">190</strong></label>
<input type="range" id="frame" min="185" max="198" step="1" value="190">
<span id="status"></span>
</div>
__PLOT__
<small>Drag to orbit 360°, scroll to zoom. Missing predictions are shown as missing.</small>
<script>
(function(){
const DATA=__DATA__;
const EDGES=__EDGES__;
const FIRST=__FIRST__,LAST=__LAST__;
const plot=document.getElementById('fb3d');
const sel=document.getElementById('method');
const slider=document.getElementById('frame');
const text=document.getElementById('ftext');
const status=document.getElementById('status');
let timer=null, frame=__INITIAL__;
function traces(points){
  if(!points)return [
    {type:'scatter3d',mode:'lines',x:[],y:[],z:[],line:{color:'#67e8f9',width:7},hoverinfo:'skip'},
    {type:'scatter3d',mode:'markers',x:[],y:[],z:[],marker:{size:4,color:'#fbbf24'},hoverinfo:'skip'}];
  const x=[],y=[],z=[];
  for(const [a,b] of EDGES){x.push(points[a][0],points[b][0],null);
                           y.push(points[a][1],points[b][1],null);
                           z.push(points[a][2],points[b][2],null);}
  return [
    {type:'scatter3d',mode:'lines',x,y,z,line:{color:'#67e8f9',width:7},hoverinfo:'skip'},
    {type:'scatter3d',mode:'markers',x:points.map(v=>v[0]),
     y:points.map(v=>v[1]),z:points.map(v=>v[2]),
     marker:{size:4,color:'#fbbf24'},hoverinfo:'skip'}];
}
function redraw(){
  slider.value=String(frame);text.textContent=String(frame);
  const points=DATA[sel.value][String(frame)];
  status.textContent=points?'21 joints':'NO DETECTION / 3D MISSING';
  Plotly.react(plot,traces(points),{...plot.layout,uirevision:'preserve-rotation'})
    .catch(e=>{status.textContent='3D rendering error: '+e;});
}
function stop(){if(timer!==null){clearInterval(timer);timer=null;}}
slider.addEventListener('input',()=>{stop();frame=Number(slider.value);redraw();});
sel.addEventListener('change',()=>{stop();redraw();});
document.getElementById('pause').addEventListener('click',stop);
document.getElementById('play').addEventListener('click',()=>{
  if(timer!==null)return;
  if(frame>=LAST)frame=FIRST-1;
  frame++;redraw();
  timer=setInterval(()=>{if(frame>=LAST){stop();return;}frame++;redraw();},350);
});
redraw();
})();
</script></main></body></html>"""
opts=''.join(f'<option value="{html.escape(k)}">{html.escape(v)}</option>' for k,v in MODEL_LABELS.items())
output=(template.replace('__PLOT__',figure_html)
    .replace('__OPTIONS__',opts)
    .replace('__DATA__',json.dumps(viewer_payload,separators=(',',':')))
    .replace('__EDGES__',json.dumps(bones))
    .replace('__FIRST__',str(FRAMES[0]))
    .replace('__LAST__',str(FRAMES[-1]))
    .replace('__INITIAL__',str(RELEASE_REFERENCE)))
viewer_html.write_text(output,encoding='utf-8')
for required in ['id="method"','id="frame"','id="play"','id="pause"','Plotly.react']:
    assert required in output,required
print('Viewer HTML:',viewer_html,round(viewer_html.stat().st_size/1e6,2),'MB')
display(HTML('<b>3D比較ビューアを生成しました。Google DriveからHTMLをダウンロードしてブラウザで開いてください。</b>'))

## 6. SOMA統合用QC出力の固定

出力データはStage 2J-1結果の再解析に限定します。手動座標もモデル別推論座標も修正しません。

In [ ]:
handoff=OUT/'stage2j1b_soma_handoff_points.csv'
finger[['frame','method','landmark_id','finger','joint','status_stage2h',
 'stage2h_x_px','stage2h_y_px','x_px','y_px','model3d_x','model3d_y','model3d_z',
 'model3d_coord_system','detected','assisted_by_stage2h_roi',
 'error_px']].to_csv(handoff,index=False)
assert hashlib.sha256(INPUT_CSV.read_bytes()).hexdigest()==source_digest,'Input changed'
report={'stage':'2J-1B','frozen_stage2j1_sha256':source_digest,
 'input_unchanged':True,'frames':FRAMES,'release_frame_visual_reference':RELEASE_REFERENCE,
 'methods':METHODS,'limitations':[
 '3D outputs have incompatible coordinate frames and are not physical 3D ground truth',
 'Reference-derived ROI is an assisted condition, not an independent detector test',
 'Jointwise errors represent 2D discrepancy to reviewed Stage2H landmarks',
 'Occluded and missing landmarks are excluded from accuracy analysis',
 'Frames with missing predictions remain missing and are not interpolated']}
(OUT/'stage2j1b_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('Outputs:',*[p.name for p in sorted(OUT.iterdir()) if p.is_file()],sep='\n - ')
print('Next Stage 2J-2: use Stage2H 2D and inspected WiLoR initial 3D to fit SOMA; verify independent held-out data.')